In [1]:
# import 
import pandas as pd
import re

In [2]:
# read data

train_df = pd.read_csv("data/train.csv")
train_df

,comment,label,label_id
0,غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است,SAD,1.0
1,بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم,HAPPY,0.0
2,غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....,SAD,1.0
3,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,SAD,1.0
4,سلام، خیلی ممنون و متشکرم,HAPPY,0.0
...,...,...,...
52105,یکی از بهترین ته چین هاییه که خوردم,HAPPY,0.0
52106,به موقع و خوب دستتون درد نکنه,HAPPY,0.0
52107,خیلی تازه و خوشمزه بود. بسته بندی شیک وعالی. خ...,HAPPY,0.0
52108,فوق العاده سریع و عالی واقعا واسه همه زود آورد...,HAPPY,0.0


In [26]:
X_test = pd.read_csv('data/test.csv')


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52110 entries, 0 to 52109
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   comment   52110 non-null  object 
 1   label     52110 non-null  object 
 2   label_id  52110 non-null  float64
dtypes: float64(1), object(2)
memory usage: 1.2+ MB


In [4]:
train_df["label"].value_counts()

label
HAPPY    26236
SAD      25874
Name: count, dtype: int64

In [5]:
train_df.columns

Index(['comment', 'label', 'label_id'], dtype='object')

In [6]:
train_df.duplicated().sum()

0

In [7]:
train_df.isnull().sum()

comment     0
label       0
label_id    0
dtype: int64

In [8]:
train_df["length"] = train_df["comment"].apply(len)
train_df

,comment,label,label_id,length
0,غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است,SAD,1.0,48
1,بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم,HAPPY,0.0,46
2,غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....,SAD,1.0,111
3,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,SAD,1.0,199
4,سلام، خیلی ممنون و متشکرم,HAPPY,0.0,25
...,...,...,...,...
52105,یکی از بهترین ته چین هاییه که خوردم,HAPPY,0.0,35
52106,به موقع و خوب دستتون درد نکنه,HAPPY,0.0,29
52107,خیلی تازه و خوشمزه بود. بسته بندی شیک وعالی. خ...,HAPPY,0.0,113
52108,فوق العاده سریع و عالی واقعا واسه همه زود آورد...,HAPPY,0.0,54


In [9]:
train_df["length"].describe()

count    52110.000000
mean        89.893667
std         77.252140
min         11.000000
25%         39.000000
50%         66.000000
75%        114.000000
max       1700.000000
Name: length, dtype: float64

In [10]:
train_df[train_df["label"] == "HAPPY"].sample(5)

,comment,label,label_id,length
7473,من رژیم دارم و ورزش میکنم به جرات میتونم بگم ت...,HAPPY,0.0,154
13632,کیفیت غذاها خیلی پایین آمده متاسفانه ولی قیمت ...,HAPPY,0.0,291
40341,کیفیت حوب بود دورچین عالی,HAPPY,0.0,25
3866,من خیلی از این فروشگاه خرید کردم. موقع خرید هم...,HAPPY,0.0,206
24270,روی جعبه‌ها مشخص نشده بود کدوم اسپایسیه و کدوم...,HAPPY,0.0,102


In [11]:
train_df[train_df["label"] == "SAD"].sample(5)

,comment,label,label_id,length
10364,برنج ته دیگ سوخته، مرغ خیلی کوچک بوی زحم کرغ,SAD,1.0,44
7454,شیرینی گردویی به شدت مونده بود طوری که از شدت ...,SAD,1.0,62
2362,نون همبرگر شیرین بود.,SAD,1.0,21
47106,یک مورد از سفارش ارسال نشده,SAD,1.0,27
36485,نان مناشب ساندویچ نبود انقدر نرم بود که از هم ...,SAD,1.0,88


In [12]:
# check urls
train_df[train_df["comment"].str.contains('http', case= False, na= False)]

,comment,label,label_id,length


In [13]:
# check mention
train_df[train_df["comment"].str.contains('@', na= False)]

,comment,label,label_id,length
24236,بسیار رستوران بد و غذای سرد و بی کیفیت!!!!! @,SAD,1.0,45
34930,از زمان سفارش تا رسیدن پیتزا ۲ ساعت و ربع!!!!!...,SAD,1.0,52
36397,سفارش با یک ساعت تاخیر به دستمان رسید چرا اینق...,SAD,1.0,67
42184,متاسفانه سفار@ ثبت شده به تعدادرونداشتن تماس گ...,SAD,1.0,59


In [14]:
# check number
train_df[train_df["comment"].str.contains(r"\d", regex= True, na= False)]

,comment,label,label_id,length
14,کیفیت پایین‌تر از پایین، در حد یه ساندویچ ۸-۹ ...,SAD,1.0,56
26,راد عزیز، متأسفانه دقت قبل در ارسال سفارش دیده...,SAD,1.0,295
29,اصلا فکر نمیکردم به این شدت غذا و کیفیتش پایین...,SAD,1.0,167
33,کیفیت بسیار بد!!! من ساندویچ ژامبون ویژه سفارش...,SAD,1.0,350
38,غذا بعد از ۱ ساعت و نیم با رفتار طلبکارانه پیک...,SAD,1.0,193
...,...,...,...,...
52088,به معنای واقعی عالی. چقدر تمیز و لایه لایه چید...,HAPPY,0.0,117
52092,طعم غذا به نسبت خوب بود منتها زمان رسید غذا خی...,HAPPY,0.0,92
52096,واقعا ۲۱ هزار تومان و مالیات ۲هزار تومنی برای ...,SAD,1.0,81
52099,سیب زمینی با قارچ و پنیر شامل ۲۰ عدد سیب زمینی...,HAPPY,0.0,217


In [15]:
# check '!'
train_df[train_df["comment"].str.contains(r"[!?؟]", regex= True, na= False)]

,comment,label,label_id,length
8,مرغی که واسه ما آوردن بو میداد انگار که مونده ...,SAD,1.0,50
14,کیفیت پایین‌تر از پایین، در حد یه ساندویچ ۸-۹ ...,SAD,1.0,56
18,بیشتر پیتزا خمیر و پنیر بود بجای مخلفات. سس هم...,SAD,1.0,59
33,کیفیت بسیار بد!!! من ساندویچ ژامبون ویژه سفارش...,SAD,1.0,350
34,شیرینی ناپلئونی خیلی معمولی بود و مزه ناپلئونی...,SAD,1.0,177
...,...,...,...,...
52052,خیلی خوب بود، چسبید!,HAPPY,0.0,20
52060,کیفیت خوب وغذا خوشمزه ولی خیلی دیر میارن. هر د...,SAD,1.0,73
52068,چندمین دفعه هستش که سفارش میدم، کیفیت نون خوبه...,HAPPY,0.0,186
52077,تعداد میگو نسبت به قیمت کم بود! ولی در کل خوشم...,HAPPY,0.0,58


# clean text

In [16]:
from src.preprocessing import preprocessing

In [18]:
train_df["clean_text"] = train_df["comment"].map(preprocessing)
train_df.sample(20)

,comment,label,label_id,length,clean_text
43760,من کیک زعفرانی یک کیلویی سفارش دادم از پشتیبان...,SAD,1.0,236,من کیک زعفرانی یک کیلویی سفارش دادم از پشتیبان...
10576,به جای ماهی، رست بیف ارسال کردند!!!,SAD,1.0,35,به جای ماهی رست بیف ارسال کردند
19816,یکم سرد شده بودش …. اگه باکس غذا داشه باشه پیک...,HAPPY,0.0,56,یکم سرد شده بودش … اگه باکس غذا داشه باشه پیک ...
42685,کیفیت غذا خیلی پایین اومده بود,SAD,1.0,30,کیفیت غذا خیلی پایین اومده بود
48763,عالی بود، اگر همین کیفیت حفظ بشه واقعا خوبه,HAPPY,0.0,43,عالی بود اگر همین کیفیت حفظ بشه واقعا خوبه
31143,سلام‌. ‌کوبیده‌بجای دو سیخ یک‌سیخ بود!!! ‌,SAD,1.0,42,سلام کوبیده‌بجای دو سیخ یک‌سیخ بود
30312,خسته نباشید کمتر از بیست دقیقه ساندویچ فیله‌ای...,HAPPY,0.0,146,خسته نباشید کمتر از بیست دقیقه ساندویچ فیله‌ای...
14204,غذاومخلفاتش ونون گرمش عالی بود ممنونم موفق باش...,HAPPY,0.0,110,غذاومخلفاتش ونون گرمش عالی بود ممنونم موفق باش...
6936,غذاهای اینجا واقعا خوبه ولی دفعه اخر ک حرید کر...,HAPPY,0.0,174,غذاهای اینجا واقعا خوبه ولی دفعه اخر ک حرید کر...
599,در ابتدا سفارش اشتباهی به ما رسید. بستنی‌ها در...,SAD,1.0,100,در ابتدا سفارش اشتباهی به ما رسید بستنی‌ها در ...


In [19]:
# tf-idf

from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
tf_idf = TfidfVectorizer()

In [21]:
X_train = tf_idf.fit_transform(train_df["clean_text"])
X_test = tf_idf.transform()
X_train.shape

(52110, 23764)

In [24]:
# dataframe

tfidf_df = pd.DataFrame(
    X_train.toarray(),
    columns= tf_idf.get_feature_names_out()
)
tfidf_df

,000,0000,0057,01,0100,011,04,07,0805,0845,...,یکیفت,یکیم,یکیه,یکیو,یگ,یگیری,یی,ییب,ییسکوئیت,ییسکوییت
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52105,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52106,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52107,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52108,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
